# Быстрый скрининг эксперимента B

Notebook сравнивает исходный четырёхканальный `BaselineUNet` и эксперимент B (`shared encoder + distance embeddings`) на фиксированных 512 train и 128 validation четвёрках. Порядок батчей, фазовая голова `π·tanh`, физика и loss одинаковы. Это быстрый скрининг на 6 эпох перед дорогим полным обучением.

In [ ]:
import csv
import gc
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

try:
    from pytorch_msssim import ssim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

# Change this only when your Kaggle Dataset has a different slug.
KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v1')
WORK_DIR = Path('/kaggle/working/experiment_b_screen')
IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)
SEED = 42
BATCH_SIZE = 16
TRAIN_SAMPLES = 512
VAL_SAMPLES = 128
EPOCHS = 6
WARMUP_EPOCHS = 1
LOG_EVERY = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(2, os.cpu_count() or 1)

def find_dataset_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_dataset_root(KAGGLE_DATASET_DIR)
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('device =', 'cuda' if torch.cuda.is_available() else 'cpu')


## Загрузчик манифестов

Синтетические каналы восстанавливаются из seed, а натуральные — из исходного JPEG и координат кропа. Каждый канал нормируется на собственный максимум, как требует уточнение организаторов.

In [ ]:
def normalise(image, eps=1e-8):
    maximum = image.amax()
    if maximum <= eps:
        raise ValueError('All-black image cannot be normalized')
    return image / maximum

def to_tensor(image):
    return normalise(torch.from_numpy(np.asarray(image.convert('L'), dtype=np.float32) / 255.0))

def draw_glyph(canvas, rng, value):
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
    font_size = int(rng.integers(22, 66))
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', size=font_size)
    except OSError:
        font = ImageFont.load_default()
    glyph = Image.new('L', (font_size * 2, font_size * 2), 0)
    glyph_draw = ImageDraw.Draw(glyph)
    box = glyph_draw.textbbox((0, 0), char, font=font)
    glyph_draw.text((-box[0], -box[1]), char, fill=value, font=font)
    box = glyph.getbbox()
    if box is not None:
        glyph = glyph.crop(box).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
        canvas.paste(glyph, (int(rng.integers(0, max(1, 129 - glyph.width))), int(rng.integers(0, max(1, 129 - glyph.height)))), glyph)

def synthetic(seed):
    rng = np.random.default_rng(int(seed))
    canvas = Image.new('L', (128, 128), 0)
    draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph']))
        value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, 124, size=2).tolist()); y0, y1 = sorted(rng.integers(4, 124, size=2).tolist())
        x1, y1 = min(127, max(x1, x0 + 8)), min(127, max(y1, y0 + 8))
        if kind == 'ellipse': draw.ellipse((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'rectangle': draw.rectangle((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'line': draw.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring': draw.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            radius = int(rng.integers(2, 10)); cx, cy = int(rng.integers(radius, 128 - radius)), int(rng.integers(radius, 128 - radius))
            draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=value)
        else: draw_glyph(canvas, rng, value)
    return to_tensor(canvas)

class QuadDataset(Dataset):
    def __init__(self, manifest):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        images = []
        for item in self.records[index]['items']:
            if item['source'] == 'synthetic':
                images.append(synthetic(item['seed']))
            else:
                path = BSDS_ROOT / item['path']
                with Image.open(path) as source:
                    x, y = item['crop']['x'], item['crop']['y']
                    images.append(to_tensor(source.crop((x, y, x + 128, y + 128))))
        quad = torch.stack(images)
        assert quad.shape == (4, 128, 128)
        return quad

full_train_set = QuadDataset(DATA_ROOT / 'manifests' / 'train.jsonl')
full_val_set = QuadDataset(DATA_ROOT / 'manifests' / 'val.jsonl')
subset_rng = np.random.default_rng(SEED + 101)
train_indices = subset_rng.choice(len(full_train_set), size=min(TRAIN_SAMPLES, len(full_train_set)), replace=False).tolist()
val_indices = subset_rng.choice(len(full_val_set), size=min(VAL_SAMPLES, len(full_val_set)), replace=False).tolist()
train_set, val_set = Subset(full_train_set, train_indices), Subset(full_val_set, val_indices)
print(f'screen train={len(train_set)}, val={len(val_set)}, example={tuple(train_set[0].shape)}')


## Baseline и shared-depth архитектура

Baseline сразу объединяет четыре изображения как входные каналы. Эксперимент B применяет один shared encoder к каждой плоскости, добавляет embeddings расстояний и объединяет признаки `1×1`-свёртками на каждом масштабе. Обе модели используют одну и ту же скалярную фазовую голову `π·tanh`.

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(), nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels, planes=4):
        super().__init__(nn.Conv2d(planes * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class SharedDepthUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.planes = 4
        channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c, self.planes) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z_mm = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z_mm - z_mm.mean()) / (z_mm.max() - z_mm.min()))
    def _fuse(self, features, fusion, batch_size):
        _, channels, height, width = features.shape
        features = features.reshape(batch_size, self.planes, channels, height, width)
        return fusion(features.reshape(batch_size, self.planes * channels, height, width))
    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.planes:
            raise ValueError(f'Expected (B, 4, H, W), got {tuple(x.shape)}')
        batch_size, _, height, width = x.shape
        features = x.reshape(batch_size * self.planes, 1, height, width)
        depths = self.depths.repeat(batch_size)
        fused = []
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features)
            features = features + embedding(depths).to(features.dtype)
            fused.append(self._fuse(features, fusion, batch_size))
            if level + 1 < len(self.encoders):
                features = self.pool(features)
        e0, e1, e2, e3, bottleneck = fused
        x = self.d3(bottleneck, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

def field_and_phase(raw):
    raw = raw.float()
    phase = math.pi * torch.tanh(raw)
    field = torch.polar(torch.ones_like(phase), phase)
    return field, phase

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(128, dtype=torch.float64) - 64) * PIXEL_PITCH; u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, field):
        outputs = []
        for kernel in self.kernels:
            propagated = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(propagated)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    ssim_values = torch.stack([ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False) for m in range(4)], dim=1)
    return (1 - cc).mean() + 1.1 * (1 - ssim_values).mean(), cc, ssim_values


## Два коротких сопоставимых запуска

Каждый режим заново получает один и тот же seed и DataLoader generator. Обучение всегда длится фиксированные 6 эпох: ранняя остановка здесь исказила бы сравнение. Прогресс печатается каждые 10 батчей. Итоговая рекомендация строится по среднему последних трёх эпох, а не по случайному лучшему значению.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('GPU is disabled. In Kaggle open Settings → Accelerator → GPU, then restart the session.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
pin_memory = device.type == 'cuda'
physics = FresnelLayer(device)

def make_loaders():
    generator = torch.Generator().manual_seed(SEED + 202)
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
    return train_loader, val_loader

def lr_multiplier(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return (1e-5 / LR) + (1 - 1e-5 / LR) * 0.5 * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def validate(model, loader):
    model.eval(); loss_total = 0.0; cc_total = torch.zeros(4, device=device); ssim_total = torch.zeros(4, device=device); count = 0
    for target in loader:
        target = target.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            raw = model(2 * target - 1)
        field, _ = field_and_phase(raw)
        loss, cc, ssim_values = doe_loss(target, physics(field))
        loss_total += loss.item() * target.shape[0]
        cc_total += cc.sum(dim=0); ssim_total += ssim_values.sum(dim=0); count += target.shape[0]
    return loss_total / count, (cc_total / count).cpu().tolist(), (ssim_total / count).cpu().tolist()

@torch.no_grad()
def phase_diagnostics(model, loader):
    model.eval(); saturation = gradient_factor = linear_tv = circular_tv = 0.0; count = 0
    for target in loader:
        target = target.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            raw = model(2 * target - 1)
        _, phase = field_and_phase(raw)
        dx, dy = phase[..., 1:] - phase[..., :-1], phase[..., 1:, :] - phase[..., :-1, :]
        saturation += (phase.abs() > 0.95 * math.pi).float().mean().item() * target.shape[0]
        gradient_factor += (1 - (phase / math.pi).square()).mean().item() * target.shape[0]
        linear_tv += 0.5 * (dx.abs().mean() + dy.abs().mean()).item() * target.shape[0]
        circular_tv += 0.5 * (torch.atan2(torch.sin(dx), torch.cos(dx)).abs().mean() + torch.atan2(torch.sin(dy), torch.cos(dy)).abs().mean()).item() * target.shape[0]
        count += target.shape[0]
    return {'boundary_fraction': saturation / count, 'tanh_gradient_factor_proxy': gradient_factor / count, 'linear_tv': linear_tv / count, 'circular_tv': circular_tv / count}

def run_experiment(architecture):
    print(f'\n===== {architecture.upper()} =====')
    set_seed(SEED)
    train_loader, val_loader = make_loaders()
    model = (BaselineUNet() if architecture == 'baseline' else SharedDepthUNet()).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    history, best_loss = [], float('inf')
    checkpoint = WORK_DIR / f'best_{architecture}.pt'
    for epoch in range(EPOCHS):
        epoch_started = time.perf_counter()
        model.train(); running = 0.0; seen = 0
        for batch_index, target in enumerate(train_loader, start=1):
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                raw = model(2 * target - 1)
            with torch.autocast(device_type=device.type, enabled=False):
                field, _ = field_and_phase(raw)
                loss, _, _ = doe_loss(target, physics(field))
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            running += loss.detach().item() * target.shape[0]; seen += target.shape[0]
            if batch_index == 1 or batch_index % LOG_EVERY == 0 or batch_index == len(train_loader):
                elapsed = time.perf_counter() - epoch_started
                eta = elapsed / batch_index * (len(train_loader) - batch_index)
                print(f'{architecture} epoch {epoch + 1}/{EPOCHS}: batch {batch_index}/{len(train_loader)}, loss={loss.item():.4f}, ETA={eta:.0f}s', flush=True)
        val_loss, cc_planes, ssim_planes = validate(model, val_loader); scheduler.step()
        row = {'epoch': epoch + 1, 'train_loss': running / seen, 'val_loss': val_loss, 'val_cc': float(np.mean(cc_planes)), 'val_ssim': float(np.mean(ssim_planes)), 'val_cc_min': float(np.min(cc_planes)), 'val_ssim_min': float(np.min(ssim_planes)), 'lr': optimizer.param_groups[0]['lr']}
        row.update({f'val_cc_z{m+1}': value for m, value in enumerate(cc_planes)})
        row.update({f'val_ssim_z{m+1}': value for m, value in enumerate(ssim_planes)})
        row['epoch_seconds'] = time.perf_counter() - epoch_started
        history.append(row); print(row, flush=True)
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({'model': model.state_dict(), 'metrics': row, 'architecture': architecture}, checkpoint)
    with (WORK_DIR / f'metrics_{architecture}.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
    state = torch.load(checkpoint, map_location=device)
    model.load_state_dict(state['model'])
    diagnostics = phase_diagnostics(model, val_loader)
    del model, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    if device.type == 'cuda': torch.cuda.empty_cache()
    return {'history': history, 'diagnostics': diagnostics, 'parameter_count': parameter_count}

architectures = ('baseline', 'shared_depth')
results = {name: run_experiment(name) for name in architectures}

def tail_mean(name, key, n=3):
    return float(np.mean([row[key] for row in results[name]['history'][-n:]]))

summary = {}
for name in architectures:
    summary[name] = {
        'tail3_cc': tail_mean(name, 'val_cc'),
        'tail3_ssim': tail_mean(name, 'val_ssim'),
        'tail3_cc_min': tail_mean(name, 'val_cc_min'),
        'tail3_ssim_min': tail_mean(name, 'val_ssim_min'),
        'tail3_epoch_seconds': tail_mean(name, 'epoch_seconds'),
        'parameter_count': results[name]['parameter_count'],
        **results[name]['diagnostics'],
    }
delta_ssim = summary['shared_depth']['tail3_ssim'] - summary['baseline']['tail3_ssim']
delta_cc = summary['shared_depth']['tail3_cc'] - summary['baseline']['tail3_cc']
delta_worst_cc = summary['shared_depth']['tail3_cc_min'] - summary['baseline']['tail3_cc_min']
delta_worst_ssim = summary['shared_depth']['tail3_ssim_min'] - summary['baseline']['tail3_ssim_min']
meaningful_gain = delta_cc >= 0.005 or delta_ssim >= 0.005
no_material_harm = delta_cc >= -0.003 and delta_ssim >= -0.003 and delta_worst_cc >= -0.005 and delta_worst_ssim >= -0.005
if meaningful_gain and no_material_harm:
    decision = 'PROMISING: repeat the small screen with a second seed before a full experiment B run.'
elif delta_cc < -0.01 or delta_ssim < -0.01 or delta_worst_cc < -0.01 or delta_worst_ssim < -0.01:
    decision = 'NOT PROMISING: shared-depth does not justify a full run in its current form.'
else:
    decision = 'BORDERLINE: repeat only this small screen with a second seed.'
summary['delta_shared_minus_baseline'] = {'ssim': delta_ssim, 'cc': delta_cc, 'worst_cc': delta_worst_cc, 'worst_ssim': delta_worst_ssim}
summary['config'] = {'train_samples': len(train_set), 'val_samples': len(val_set), 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'seed': SEED, 'lr': LR, 'zs': ZS}
summary['decision'] = decision
(WORK_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, color in [('baseline', 'tab:blue'), ('shared_depth', 'tab:orange')]:
    epochs = [row['epoch'] for row in results[name]['history']]
    axes[0].plot(epochs, [row['val_loss'] for row in results[name]['history']], marker='o', label=name, color=color)
    axes[1].plot(epochs, [row['val_cc'] for row in results[name]['history']], marker='o', label=name, color=color)
    axes[2].plot(epochs, [row['val_ssim'] for row in results[name]['history']], marker='o', label=name, color=color)
for axis, title in zip(axes, ['Validation loss', 'Validation CC', 'Validation SSIM']):
    axis.set_title(title); axis.set_xlabel('epoch'); axis.grid(alpha=.3); axis.legend()
plt.tight_layout(); plt.savefig(WORK_DIR / 'comparison.png', dpi=160); plt.show()
print('Decision:', decision)
print('Saved to:', WORK_DIR)
